# Evolution of agents - multi-agent

Agents working together on a single task.
<img src="./images/multi_agent.png" alt="Agent with Actions" style="max-height: 300px;" />

**Description:**
This notebook explores multi-agent orchestration, where several agents collaborate to solve complex tasks. It demonstrates agent-to-agent communication, coordination, and division of labor, showing how specialized agents can work together for more powerful solutions.

⚠️  IMPORTANT - Playwright Connection Token Expiration:
The Playwright connection uses an access token that expires after ~75 minutes.

⚠️ ** Use of access token is not recommended. This has been done for convenience of agent creation and connection for this demo.
Do not use this pattern. ** ⚠️

If you're running this script more than an hour after deployment (or days later),
and you want to use the Playwright agent then you may need to refresh the connection by running:
    .\infra\scripts\add-playwright-connection.ps1

If you see authentication or connection errors, run the script above for a fresh token.

Windows Users:

    Usage:
        python 6-mcp-windows.py

Mac/Linux Users:
    Usage:
        bash
            ./infra/scripts/add-playwright-connection.sh

In [1]:
# Example: Inference using Semantic Kernel
from semantic_kernel import Kernel
import os
from setup import (
    get_project_client,
    create_agent,
    create_weather_openapi_tool,
    get_connection_by_name,
    test_agent,
)
from AzureStandardLogicAppTool import create_logic_app_tools
from azure.ai.agents.models import BingGroundingTool
from azure.ai.agents.models import ConnectedAgentTool

client = await get_project_client()

logic_app_tools = create_logic_app_tools(
    logic_app_subscription_id=os.environ.get("LOGIC_APP_SUBSCRIPTION_ID"),
    logic_app_resource_group=os.environ.get("LOGIC_APP_RESOURCE_GROUP"),
    logic_app_name=os.environ.get("LOGIC_APP_NAME"),
    foundry_subscription_id=os.environ.get("AZURE_AI_FOUNDRY_SUBSCRIPTION_ID"),
    foundry_resource_group=os.environ.get("AZURE_AI_FOUNDRY_RESOURCE_GROUP"),
    foundry_foundry_name=os.environ.get("AZURE_AI_FOUNDRY_NAME"),
    foundry_project_name=os.environ.get("AZURE_AI_FOUNDRY_PROJECT_NAME"),
)
current_date_tool = next(
    (tool for tool in logic_app_tools if "time" in tool.definitions[0]["openapi"].name),
    None,
)

kernel = Kernel()

office_agent = await create_agent(
    agent_name="Office365-agent",
    agent_instructions="You are a helpful assistant. You use Office 365 tools to send emails, schedule events and check calendars. Make sure you know the current date! It doesn't make sense to schedule events in the past.",
    client=client,
    kernel=kernel,
    tools=[def_ for tool in logic_app_tools for def_ in tool.definitions],
)

weather_tool = create_weather_openapi_tool()

weather_agent = await create_agent(
    agent_name="Weather-agent",
    agent_instructions="You are a helpful assistant. You use weather tools to provide weather information.",
    client=client,
    kernel=kernel,
    tools=(
        weather_tool.definitions + current_date_tool.definitions
        if current_date_tool
        else []
    ),
)

bing_grounding = BingGroundingTool(
    connection_id=await get_connection_by_name(client, "bing")
)
news_agent = await create_agent(
    agent_name="Bing-agent",
    agent_instructions="Use the Bing grounding tool to answer the user's question.",
    client=client,
    kernel=kernel,
    tools=(
        bing_grounding.definitions + current_date_tool.definitions
        if current_date_tool
        else []
    ),
)

# Uncomment if using the blog agent
# blog_url = os.environ.get("BLOG_URL", None)
# print(f"Blog URL: {blog_url}")
# playwright_agent = await create_agent(
#     agent_name="Playwright-agent",
#     agent_instructions=f"You're a blog manager. You read blog posts and post new ones on {blog_url} using browser automation.",
#     client=client,
#     kernel=kernel,
#     tools=[
#         {
#             "type": "browser_automation",
#             "browser_automation": {
#                 "connection": {"id": await get_connection_by_name(client, "Playwright")}
#             },
#         }
#     ],
# )

connected_agents = [
    ConnectedAgentTool(
        id=office_agent.id,
        name="calendar_email_agent",
        description="""
Calendar agent can:
* check calendar of the user
* schedule new events (book running time) by providing event details (subject, start time, end time)
* email users (to, subject, body)
* check current date/time
                       """,
    ),
    ConnectedAgentTool(
        id=weather_agent.id,
        name="weather_agent",
        description="""
Weather agent can:
* provide current weather information for provided location
* provide weather forecasts for provided location
                       """,
    ),
# Uncomment if using the blog agent   
#  ConnectedAgentTool(
#         id=playwright_agent.id,
#         name="blog_agent",
#         description=f"""
# Blog agent can manage running blog {blog_url} using playwright tool
# * read blog posts
# * post new blog entries
#                        """,
#     ),
    ConnectedAgentTool(
        id=news_agent.id,
        name="news_agent",
        description="""
News agent can perform web research using bing grounding tool
                       """,
    ),
]

main_agent = await create_agent(
    agent_name="Main-agent",
    agent_instructions="""
You are **AdvisorGPT**, a helpful, professional agent supporting user queries. 
Your main objective is to fully resolve each user query related to health, fitness and running, using only verified information and the tools listed below.

---

## **Workflow**

1. **Plan Before Action:**  
   Before using any agent, briefly outline your step-by-step plan to address the user’s request.

2. **Agent Use:**  
   Use the most relevant agent(s) for accurate, up-to-date information
   If agent results are unclear or incomplete, use additional agents or ask the user for clarification.

3. **Reflect After Each Agent Call:**  
   - Did you get all the info needed?  
   - What’s the next logical step?  
   - Is further clarification needed?  
   - Is the problem fully solved?

4. **Clear Output:**  
   - Use clear sections, bullet points, numbered steps, or tables where helpful.
   - Use a joyful, helpful tone, feel free to use emojis

5. **If Unsure:**  
   - Never guess. Use Agents to find the answer or ask the user to clarify.

6. **Prohibited Topics:**  
   - If the request is outside of domain supported by connected agents, politely redirect the user:
   - Example: “I’m sorry, but I can’t discuss that topic. Is there something OTIS-related I can assist you with?”

7. **Use Date/Time Tool:**  
   - Always use `get-date when current or time-bounded information is needed.

8. **Use Location Information:**
   - Ask user for location information when relevant, especially for weather and calendar events.

# Uncomment if using blog agent
# 9. **Use blog for memory**
#    - Utilize the blog_agent check for previous posts or relevant information before scheduling new runs.

---

## **Available Agents**

| Agent Name                           | Purpose/Usage                                 | Available Tools |
|--------------------------------------|-----------------------------------------------|-----------------|
| calendar_email_agent                 | calendar management and email communication   | create_event, get_events, email_me, get_current_time |
| weather_agent                        | current weather and forecast                  | WeatherAPI, get_current_time |
| news_agent                           | bing search for current information           | bing grounding search |
# Uncomment if using blog agent
# | blog_agent                           | running blog management                       | Executing actions on the running blog |

**Before Using an Agent:**  
- “Let me verify your information to assist you.”  
- “One moment, I’ll check that for you.”

**After Using an Agent:**  
- “Here’s what I found: [summary]”  
- “Based on the data, here are your next steps…”

## **Key Guidelines**

- Plan first, act second.
- Reflect after every action.
- Never guess—always use tools or clarify.
- Structure responses for clarity.
- Only finish when the issue is fully resolved.

    """,
    client=client,
    kernel=kernel,
    tools=[def_ for tool in connected_agents for def_ in tool.definitions],
)

Workflows: [
  {
    "name": "create_event",
    "definition_href": "https://logic-apps-i4f5yo2z5kt3c-i4f5yo2z5kt3c.azurewebsites.net/admin/vfs/site/wwwroot/create_event/workflow.json",
    "href": "https://logic-apps-i4f5yo2z5kt3c-i4f5yo2z5kt3c.azurewebsites.net/runtime/webhooks/workflow/api/management/workflows/create_event",
    "kind": "Stateful",
    "triggers": {
      "When_a_HTTP_request_is_received": {
        "type": "Request",
        "kind": "Http"
      }
    },
    "isDisabled": false,
    "health": {
      "state": "Healthy"
    }
  },
  {
    "name": "email_me",
    "definition_href": "https://logic-apps-i4f5yo2z5kt3c-i4f5yo2z5kt3c.azurewebsites.net/admin/vfs/site/wwwroot/email_me/workflow.json",
    "href": "https://logic-apps-i4f5yo2z5kt3c-i4f5yo2z5kt3c.azurewebsites.net/runtime/webhooks/workflow/api/management/workflows/email_me",
    "kind": "Stateful",
    "triggers": {
      "When_a_HTTP_request_is_received": {
        "type": "Request",
        "kind": "Http"
   

## Test the Logic Apps


In [15]:
# Direct HTTP call to the get_events Logic App
import requests
import json
from datetime import datetime, timedelta, timezone

print("🔍 Checking Logic App connectivity...")
print("=" * 50)

# Find the get_events tool using the same pattern as current_date_tool
get_events_tool = next(
    (tool for tool in logic_app_tools if "get_events" in tool.definitions[0]["openapi"].name),
    None,
)

if get_events_tool:
    # Extract the callback URL from the OpenAPI spec
    openapi_def = get_events_tool.definitions[0]["openapi"]
    callback_url = openapi_def.spec["servers"][0]["url"]
    print(f"📋 Tool name: {openapi_def.name}")
    print(f"🔗 Callback URL: {callback_url[:80]}...")

    # Prepare request body with required parameters
    now = datetime.now(timezone.utc)
    end_date = now + timedelta(days=1)  # Get events for the next day
    
    request_body = {
        "startDateTime": now.strftime("%Y-%m-%dT%H:%M:%SZ"),  # ISO 8601 format
        "endDateTime": end_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "maxResults": 10
    }
    print(f"📤 Request body: {request_body}")
    print("=" * 50)

    # Make the HTTP POST request
    response = requests.post(
        callback_url,
        json=request_body,
        headers={"Content-Type": "application/json"}
    )

    print(f"📊 Status: {response.status_code}")
    
    if response.status_code == 200:
        print("✅ Logic App is ready to use!")
        if response.headers.get('content-type', '').startswith('application/json'):
            # Pretty print JSON and truncate to 500 chars
            response_str = json.dumps(response.json(), indent=2)
            print(f"📥 Response (truncated): {response_str[:500]}{'...' if len(response_str) > 500 else ''}")
        else:
            print(f"📥 Response: {response.text[:500]}")
    elif response.status_code == 401 or response.status_code == 403:
        print("❌ Authorization error!")
        print("⚠️  The Office 365 connection needs to be authorized in Azure Portal.")
        print("👉 Go to: Azure Portal → Logic Apps → Your Logic App → API Connections → Authorize")
    else:
        print(f"⚠️  Unexpected response (Status: {response.status_code})")
        response_text = response.text
        if "AADSTS" in response_text or "authorization" in response_text.lower():
            print("❌ Connection authorization required!")
            print("👉 Go to Azure Portal → Logic Apps → API Connections → Authorize the Office 365 connection")
        else:
            print(f"📥 Response: {response_text[:500]}")
else:
    print("❌ get_events tool not found in logic_app_tools")
    print("📋 Available tools:")
    for tool in logic_app_tools:
        print(f"  - {tool.definitions[0]['openapi'].name}")

🔍 Checking Logic App connectivity...
📋 Tool name: get_events
🔗 Callback URL: https://logic-apps-i4f5yo2z5kt3c-i4f5yo2z5kt3c.azurewebsites.net:443/api/get_eve...
📤 Request body: {'startDateTime': '2026-01-05T16:54:35Z', 'endDateTime': '2026-01-06T16:54:35Z', 'maxResults': 10}
📊 Status: 200
✅ Logic App is ready to use!
📥 Response (truncated): {
  "events": "[{\"subject\":\"Medical check - labs\",\"start\":\"2026-01-06T13:00:00.0000000\",\"end\":\"2026-01-06T14:00:00.0000000\",\"startWithTimeZone\":\"2026-01-06T13:00:00+00:00\",\"endWithTimeZone\":\"2026-01-06T14:00:00+00:00\",\"body\":\"<html>\\r\\n<head>\\r\\n<meta http-equiv=\\\"Content-Type\\\" content=\\\"text/html; charset=utf-8\\\">\\r\\n<meta name=\\\"x-apple-disable-message-reformatting\\\">\\r\\n<meta name=\\\"viewport\\\" content=\\\"width=device-width, initial-scale=1.0\\\...


In [5]:
thread = None
user_input = """
    I want to go for a run tomorrow in Cary, NC. Check the weather forecast and my calendar. If the weather is okay, schedule the run.  
Email me with a summary of everything you've done to piotrkarpala@microsoft.com"
"""
thread = await test_agent(client, main_agent, user_input, thread)

Starting agent conversation with message: 
    I want to go for a run tomorrow in Cary, NC. Check the weather forecast and my calendar. If the weather is okay, schedule the run.  
Email me with a summary of everything you've done to piotrkarpala@microsoft.com"

Processing response #1
Agent: **Step-by-step Plan:**

1. Check the weather forecast for Cary, NC for tomorrow.
2. Check your calendar for any existing events tomorrow.
3. If the weather is suitable for running, and your calendar is free, schedule a run.
4. Email you a summary of all actions taken.

Let me verify your information to assist you. One moment, I’ll check the weather and your calendar.
Processing response #2
Agent: Here’s a summary of everything I’ve done for you:

---

**1. Weather Forecast for Cary, NC (Tomorrow, Jan 6th):**
- Mostly cloudy, high of 55°F (13°C), low of 36°F (2°C)
- No significant chance of rain
- Winds: 5–15 mph, gusts up to 18 mph
- Morning fog/mist, but clear for running after 9 AM【message_idx:sea

In [ ]:
# Optional - Run to test playwright agent - Must configure manually
# user_input = """
#     Read the latest blog post on my running blog and write a new blog post about preparing for a marathon.
# """
# thread = await test_agent(client, playwright_agent, user_input, thread)

In [ ]:
# uncomment for follow up
# thread = await test_agent(client, main_agent, "try again", thread)